In [3]:
import sys
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. Chỉ đường cho Python ra thư mục gốc TTCS
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Import các hàm từ src của nhóm mình
from src.utils import evaluate_model
from src.data_loader import load_and_preprocess

print("--- Đang nạp mô hình và dữ liệu CNN ---")

# 3. Tải lại mô hình và Label Encoder (Nhớ dấu ../)
model = load_model('../models/best_cnn_model.h5', compile=False)
le = joblib.load('../models/label_encoder_cnn.pkl')

# 4. Tải dữ liệu Test để đánh giá
_, X_test_s, _, y_test, _, _ = load_and_preprocess('../data/Dataset-Unicauca-Version2-87Atts.csv')
X_test_3d = np.expand_dims(X_test_s, axis=2)

# 5. Dự đoán và Xử lý nhãn
print("--- Đang xử lý dự đoán... ---")
y_pred = model.predict(X_test_3d)
y_pred_classes = np.argmax(y_pred, axis=1)

# Lọc bỏ nhãn lạ (ví dụ nhãn 39)
valid_mask = np.isin(y_test, le.classes_)
y_test_filtered = y_test[valid_mask]
y_pred_filtered = y_pred_classes[valid_mask]

# Chuyển đổi về dạng số
y_test_fixed = le.transform(y_test_filtered)

# Lấy danh sách các nhãn thực tế xuất hiện (dạng số)
actual_labels = np.unique(y_test_fixed)

# ÉP KIỂU VỀ STRING: Đảm bảo actual_names là danh sách các chuỗi tên App
actual_names = [str(le.classes_[i]) for i in actual_labels]

print("\n--- KẾT QUẢ F1-SCORE CNN---")
from sklearn.metrics import classification_report

# Chạy lệnh này là bảng hiện ra ngay!
report = classification_report(y_test_fixed, y_pred_filtered, labels=actual_labels, target_names=actual_names)
print(report)

--- Đang nạp mô hình và dữ liệu CNN ---
--- Đang xử lý dự đoán... ---
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step

--- KẾT QUẢ F1-SCORE CNN---
              precision    recall  f1-score   support

           0       0.78      0.44      0.56      1400
           1       0.34      0.47      0.40       139
           2       0.05      0.26      0.08        23
           3       0.01      0.11      0.03        19
           7       0.00      0.00      0.00         1
           8       0.72      0.40      0.51       264
           9       0.82      0.86      0.84       145
          10       0.00      0.00      0.00         1
          11       0.47      0.95      0.62        21
          12       0.83      0.72      0.78       431
          13       0.02      0.43      0.04         7
          14       0.01      0.09      0.02        22
          16       0.87      0.71      0.78       485
          17       0.00      0.00      0.00         1
          18       0.00      0.00      0.00   

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [1]:
import sys
import os
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# 1. Xử lý đường dẫn hệ thống (Cực kỳ quan trọng trong Notebook)
# Lấy thư mục hiện tại của Notebook
current_dir = os.getcwd()
# Nếu đang ở trong folder 'notebooks', lùi ra 1 cấp để thấy 'src', 'models', 'data'
root_path = os.path.abspath(os.path.join(current_dir, '..')) 

if root_path not in sys.path:
    sys.path.append(root_path)

from src.data_loader import load_and_preprocess

print("--- Đang nạp mô hình và dữ liệu LSTM ---")

# 2. Cấu hình đường dẫn file (Dùng biến để dễ sửa)
MODEL_PATH = os.path.join(root_path, 'models', 'best_lstm_model.h5')
LE_PATH = os.path.join(root_path, 'models', 'label_encoder_lstm.pkl')
DATA_PATH = os.path.join(root_path, 'data', 'Dataset-Unicauca-Version2-87Atts.csv')

# 3. Tải mô hình và Label Encoder
# Thêm compile=False để tránh lỗi Custom Loss (Focal Loss)
model_lstm = load_model(MODEL_PATH, compile=False)

if os.path.exists(LE_PATH):
    le = joblib.load(LE_PATH)
    print(f"✅ Đã nạp thành công: {os.path.basename(LE_PATH)}")
else:
    # Cảnh báo mạnh hơn nếu không tìm thấy file pkl đúng của LSTM
    print("⚠️ CẢNH BÁO: Không tìm thấy label_encoder_lstm.pkl!")
    le_cnn_path = os.path.join(root_path, 'models', 'label_encoder_cnn.pkl')
    le = joblib.load(le_cnn_path)
    print("Sử dụng tạm label_encoder_cnn.pkl (Cần cẩn thận kết quả)")

# 4. Tải và Reshape dữ liệu
# Lưu ý: Notebook thường tốn RAM, nếu bị đơ máy, hãy giảm sample_size ở đây
_, X_test_s, _, y_test, _, _ = load_and_preprocess(DATA_PATH)
X_test_3d = np.expand_dims(X_test_s, axis=2)

print(f"--- Đang xử lý dự đoán cho {len(X_test_3d)} mẫu... ---")
y_pred = model_lstm.predict(X_test_3d, batch_size=128) # Thêm batch_size để chạy nhanh hơn
y_pred_classes = np.argmax(y_pred, axis=1)

# 5. Xử lý lọc nhãn và tạo báo cáo
valid_mask = np.isin(y_test, le.classes_)
y_test_filtered = y_test[valid_mask]
y_pred_filtered = y_pred_classes[valid_mask]

y_test_fixed = le.transform(y_test_filtered)
actual_labels = np.unique(y_test_fixed) 
actual_names = [str(le.classes_[i]) for i in actual_labels]

print("\n--- KẾT QUẢ F1-SCORE LSTM (Nâng cao với Focal Loss) ---")
report = classification_report(y_test_fixed, y_pred_filtered, 
                               labels=actual_labels, 
                               target_names=actual_names)
print(report)

--- Đang nạp mô hình và dữ liệu LSTM ---
✅ Đã nạp thành công: label_encoder_lstm.pkl


Exception in callback BaseAsyncIOLoop._handle_events()
handle: <Handle BaseAsyncIOLoop._handle_events()>
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 208, in _handle_events
    handler_func(fileobj, events)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py", line 600, in _handle_events
    self._handle_recv()
    ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py", line 629, in _handle_recv
    self._run_callback(callback, msg)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\zmq\eventloo

--- Đang xử lý dự đoán cho 60000 mẫu... ---
469/469 ━━━━━━━━━━━━━━━━━━━━ 25s 52ms/step

--- KẾT QUẢ F1-SCORE LSTM (Nâng cao với Focal Loss) ---
              precision    recall  f1-score   support

           0       0.88      0.32      0.46      1400
           1       0.24      0.23      0.24       139
           2       0.08      0.26      0.13        23
           3       0.03      0.05      0.04        19
           7       0.00      0.00      0.00         1
           8       0.65      0.31      0.42       264
           9       0.38      0.88      0.53       145
          10       0.00      0.00      0.00         1
          11       0.46      0.90      0.61        21
          12       0.85      0.69      0.76       431
          13       0.08      0.43      0.14         7
          14       0.00      0.00      0.00        22
          16       0.86      0.61      0.71       485
          17       0.00      0.00      0.00         1
          18       0.00      0.00      0.00  

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [ ]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. Xử lý đường dẫn
current_dir = os.getcwd()
root_path = os.path.abspath(os.path.join(current_dir, '..')) 
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data_loader import load_and_preprocess

print("--- [XGBoost] Đang nạp mô hình và dữ liệu ---")

# 2. Định nghĩa đường dẫn
MODEL_PATH = os.path.join(root_path, 'models', 'best_xgboost_model.json')
LE_FINAL_PATH = os.path.join(root_path, 'models', 'label_encoder_xgboost.pkl')
LE_INTERNAL_PATH = os.path.join(root_path, 'models', 'label_encoder_xgboost_internal.pkl')
DATA_PATH = os.path.join(root_path, 'data', 'Dataset-Unicauca-Version2-87Atts.csv')

# 3. Tải mô hình và Encoder
model_xgb = XGBClassifier()
model_xgb.load_model(MODEL_PATH)
le_final = joblib.load(LE_FINAL_PATH)
le_xg = joblib.load(LE_INTERNAL_PATH)

# 4. Tải và chuẩn bị dữ liệu Test
_, X_test_s, _, y_test, _, _ = load_and_preprocess(DATA_PATH)
X_test_2d = X_test_s.reshape(X_test_s.shape[0], -1)

print(f"--- [XGBoost] Đang xử lý dự đoán cho {len(X_test_2d)} mẫu... ---")
y_pred_encoded = model_xgb.predict(X_test_2d)

# 5. LỌC NHÃN: Đảm bảo dữ liệu test thuộc tập mà mô hình đã học
mask = np.isin(y_test, le_xg.classes_)
y_test_filtered = y_test[mask]
y_pred_filtered = y_pred_encoded[mask]

# Chuyển đổi nhãn test về hệ 0, 1, 2... mà XGBoost hiểu
y_test_encoded = le_xg.transform(y_test_filtered)

# 6. LẤY TÊN APP CHUẨN (Khớp với thứ tự lớp của XGBoost)
# le_xg.classes_ chứa giá trị gốc (ví dụ: 1, 2, 5, 20...)
# le_final.classes_ chứa tên ứng dụng tương ứng
actual_names = [str(le_final.classes_[int(c)]) for c in le_xg.classes_]

print("\n--- KẾT QUẢ F1-SCORE XGBOOST (With Sample Weighting) ---")
# labels=np.arange(len(actual_names)) đảm bảo khớp với encoder nội bộ
report = classification_report(
    y_test_encoded, 
    y_pred_filtered, 
    labels=np.arange(len(actual_names)),
    target_names=actual_names,
    zero_division=0
)
print(report)

--- Đang nạp mô hình và dữ liệu XGBoost ---
--- Đang xử lý dự đoán XGBoost... ---

--- KẾT QUẢ F1-SCORE XGBOOST CHO HIẾU ---
              precision    recall  f1-score   support

           0       0.82      0.49      0.62      1400
           1       0.46      0.40      0.43       139
           2       0.07      0.13      0.09        23
           3       0.10      0.16      0.12        19
           8       0.87      0.44      0.58       264
           9       0.91      0.88      0.90       145
          11       0.57      0.76      0.65        21
          12       0.96      0.72      0.82       431
          13       0.21      0.43      0.29         7
          14       0.00      0.00      0.00        22
          16       0.92      0.73      0.82       485
          18       0.00      0.00      0.00         5
          19       0.45      0.07      0.12       702
          20       0.73      0.81      0.77     16112
          21       0.00      0.00      0.00        10
          

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape